# EmoBERTa-style RoBERTa on IEMOCAP (6-way)

This notebook trains a **RoBERTa** classifier on **IEMOCAP 6-way** labels using an **EmoBERTa-style context string** where the **target utterance is surrounded by exactly two `</s>` tokens**.

## What you should add for a GitHub repo
- Put your CSVs under `data/` (not `/content/...`) and keep data **out of git** (see `.gitignore`).
- Add `requirements.txt` (or `environment.yml`) instead of `!pip install ...`.
- Add a `README.md` explaining:
  - how to obtain IEMOCAP (license restrictions apply),
  - how to create the train/val/test CSVs,
  - how to run training and reproduce results.


## 0. Imports & environment

In [ ]:
# If you're running this from a clean environment, install deps once:
#   pip install -r requirements.txt
#
# (In Colab you can alternatively run: !pip install -r requirements.txt)

import os

# Restrict this process to a single physical GPU (shared machine — only 1 of
# several A100s is available to us). Must be set before `import torch` /
# any CUDA init, since it controls which physical device maps to cuda:0.
os.environ["CUDA_VISIBLE_DEVICES"] = "6"

import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import optuna

from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    set_seed,
)

# Optional: version printout for reproducibility
import transformers, datasets
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("torch:", torch.__version__)
print("visible GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("using:", torch.cuda.get_device_name(0))


## 1. Configuration

In [ ]:
# =====================
# CONFIG (IEMOCAP 6-way)
# =====================

# Project layout
PROJECT_ROOT = Path(".").resolve()
OUTPUT_DIR = Path('/storage/data3/perikos/kontopoulos/checkpoints/emoberta_iemocap_large/')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Pre-built EmoBERTa-style context CSVs (target-SEP-only construction, already run once)
TRAIN_CTX = '/storage/data3/perikos/kontopoulos/erc-using-llms-and-explainabillity-methods-/Datasets /IEMOCAP/train_constructed_targetSEPonly_spaces.csv'
VAL_CTX   = '/storage/data3/perikos/kontopoulos/erc-using-llms-and-explainabillity-methods-/Datasets /IEMOCAP/val_constructed_targetSEPonly_spaces.csv'
TEST_CTX  = '/storage/data3/perikos/kontopoulos/erc-using-llms-and-explainabillity-methods-/Datasets /IEMOCAP/test_constructed_targetSEPonly_spaces.csv'

# IEMOCAP 6 emotions
LABELS = ["neutral", "frustration", "sadness", "anger", "excited", "happiness"]
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}

# Model
MODEL_BASE = "roberta-large"

# Paper-like constants
WEIGHT_DECAY = 0.01
EPOCHS = 7
WARMUP_RATIO = 0.20
LR_SCHED = "linear"

# Optuna: tune ONLY peak LR
N_TRIALS = 5
LR_LOW, LR_HIGH = 1e-6, 1e-4

# Training defaults
MAX_LEN = 512
BATCH_TRAIN = 8
BATCH_EVAL  = 16
GRAD_ACCUM  = 1

# Reproducibility / reporting
SEED = 42
SEEDS_FINAL = [42, 43, 44, 45, 46]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)
print("LABELS:", LABELS)

# Quick sanity check (helps avoid confusing FileNotFound errors)
for p in [TRAIN_CTX, VAL_CTX, TEST_CTX]:
    if not Path(p).exists():
        print(f"⚠️ Missing file: {p}")


## 2. Tokenizer, collator, metrics

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_BASE, use_fast=True)
collator = DataCollatorWithPadding(tokenizer=tok)

def compute_metrics(eval_pred):
    """Compatibility-friendly metrics (works across Trainer versions)."""
    preds, labels = eval_pred
    # Some HF versions return a tuple(preds,) for logits
    if isinstance(preds, (tuple, list)):
        preds = preds[0]
    y_pred = np.argmax(preds, axis=1)
    return {
        "acc": accuracy_score(labels, y_pred),
        "weighted_f1": f1_score(labels, y_pred, average="weighted"),
        "macro_f1": f1_score(labels, y_pred, average="macro"),
    }

print("CLS:", tok.cls_token, tok.cls_token_id, "SEP:", tok.sep_token, tok.sep_token_id)


## 3. Load pre-built context CSVs & tokenize

In [ ]:
# ==========================
# Load pre-built EmoBERTa-style context CSVs -> tokenized HF Datasets
# ==========================
# TRAIN_CTX / VAL_CTX / TEST_CTX already contain the "target-SEP-only" context
# strings (dialogue_id, utterance_id, label_id, label, context_text_raw)
# produced by a previous run of this notebook's context builder.
#
# RoBERTa's fast tokenizer recognizes the literal <s> / </s> tags embedded in
# context_text_raw as real special tokens, so we tokenize with
# add_special_tokens=False to avoid ending up with duplicated specials
# (verified: default add_special_tokens=True yields "<s><s> ... </s>" and an
# extra trailing </s>; add_special_tokens=False reproduces the original
# "<s> LEFT </s> TARGET </s> RIGHT" structure exactly).

def load_context_csv(path: str) -> pd.DataFrame:
    print(f"--- Loading {path} ---")
    df = pd.read_csv(path)
    print("Shape:", df.shape)
    df["label"] = df["label"].astype(str).str.strip().str.lower()
    missing = sorted(set(df["label"].unique()) - set(LABELS))
    if missing:
        raise ValueError(f"Unexpected labels in {path}: {missing}")
    print("Label counts:\n", df["label"].value_counts())
    return df

train_ctx_df = load_context_csv(TRAIN_CTX)
val_ctx_df   = load_context_csv(VAL_CTX)
test_ctx_df  = load_context_csv(TEST_CTX)

print("Rows:", len(train_ctx_df), len(val_ctx_df), len(test_ctx_df))


def build_dataset_from_context_csv(df: pd.DataFrame, tokenizer, max_length: int = 512) -> Dataset:
    """Tokenize pre-built context_text_raw strings into input_ids/attention_mask/labels."""
    enc = tokenizer(
        df["context_text_raw"].tolist(),
        add_special_tokens=False,
        truncation=True,
        max_length=max_length,
    )
    return Dataset.from_dict({
        "dialogue_id": df["dialogue_id"].astype(str).tolist(),
        "utterance_id": df["utterance_id"].astype(str).tolist(),
        "input_ids": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "labels": [label2id[l] for l in df["label"]],
    })


train_ds_full = build_dataset_from_context_csv(train_ctx_df, tok, max_length=MAX_LEN)
val_ds_full   = build_dataset_from_context_csv(val_ctx_df,   tok, max_length=MAX_LEN)
test_ds_full  = build_dataset_from_context_csv(test_ctx_df,  tok, max_length=MAX_LEN)

print("Sizes:", len(train_ds_full), len(val_ds_full), len(test_ds_full))

# Sanity check: decode one example to confirm exactly 1x <s> and 2x </s>
example_ids = train_ds_full[0]["input_ids"]
print("\nDECODED (first 140 tokens):")
print(tok.decode(example_ids[:140], skip_special_tokens=False))
print(f"n <s>: {example_ids.count(tok.cls_token_id)} | n </s>: {example_ids.count(tok.sep_token_id)}")


## 4. Hyperparameter search (Optuna)

In [ ]:
def objective(trial):
    """Optuna objective: minimize validation loss by tuning ONLY learning rate."""
    set_seed(SEED)

    lr = trial.suggest_float("lr", LR_LOW, LR_HIGH, log=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_BASE,
        num_labels=len(LABELS),
        label2id=label2id,
        id2label=id2label,
    ).to(DEVICE)

    args = TrainingArguments(
        output_dir=str(OUTPUT_DIR / f"optuna_lr_trial_{trial.number}"),
        eval_strategy="epoch",   # renamed from `evaluation_strategy` (removed in transformers 4.57)
        save_strategy="no",

        learning_rate=lr,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_TRAIN,
        per_device_eval_batch_size=BATCH_EVAL,
        gradient_accumulation_steps=GRAD_ACCUM,

        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type=LR_SCHED,

        fp16=torch.cuda.is_available(),
        report_to="none",
        seed=SEED,
        logging_steps=200,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds_full,
        eval_dataset=val_ds_full,
        data_collator=collator,
        tokenizer=tok,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    out = trainer.evaluate(val_ds_full)
    return out["eval_loss"]  # minimize cross-entropy


In [ ]:
# Tip for GitHub reproducibility: seed Optuna's sampler
study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
)
study.optimize(objective, n_trials=N_TRIALS)

best_lr = study.best_params["lr"]
print("Best lr:", best_lr)
print("Best val loss:", study.best_value)


## 5. Final training across multiple seeds + evaluation

In [ ]:
rows = []

# ---------- callback: save at end of each epoch ----------
class SaveByEpochCallback(TrainerCallback):
    """Extra per-epoch saving (separate from Trainer's own save_strategy)."""
    def __init__(self, out_root: Path, tokenizer):
        self.out_root = Path(out_root)
        self.tokenizer = tokenizer
        self.out_root.mkdir(parents=True, exist_ok=True)

    def on_epoch_end(self, args, state, control, **kwargs):
        model = kwargs["model"]
        ep = state.epoch
        ep_i = int(round(ep)) if ep is not None else 0

        save_dir = self.out_root / f"epoch_{ep_i:02d}"
        save_dir.mkdir(parents=True, exist_ok=True)

        model.save_pretrained(save_dir)
        self.tokenizer.save_pretrained(save_dir)
        print(f"✅ Saved epoch checkpoint to: {save_dir}")
        return control


for seed in SEEDS_FINAL:
    print("\n" + "=" * 20, "SEED", seed, "=" * 20)
    set_seed(seed)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_BASE,
        num_labels=len(LABELS),
        label2id=label2id,
        id2label=id2label,
    ).to(DEVICE)

    out_dir = OUTPUT_DIR / f"roberta_iemocap_final_seed{seed}"

    # Keep epoch checkpoints separate (so Trainer's checkpoint cleanup doesn't delete them)
    epoch_root = OUTPUT_DIR / f"epoch_checkpoints_seed{seed}"
    if epoch_root.exists():
        shutil.rmtree(epoch_root)
    epoch_root.mkdir(parents=True, exist_ok=True)

    epoch_saver = SaveByEpochCallback(epoch_root, tok)

    args = TrainingArguments(
        output_dir=str(out_dir),
        eval_strategy="epoch",   # renamed from `evaluation_strategy` (removed in transformers 4.57)
        save_strategy="epoch",
        save_total_limit=2,

        load_best_model_at_end=True,
        metric_for_best_model="weighted_f1",
        greater_is_better=True,

        learning_rate=best_lr,
        # NOTE: You used 7 here originally; consider moving this to config so it's not "magic".
        num_train_epochs=7,
        per_device_train_batch_size=BATCH_TRAIN,
        per_device_eval_batch_size=BATCH_EVAL,
        gradient_accumulation_steps=GRAD_ACCUM,

        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type=LR_SCHED,

        fp16=torch.cuda.is_available(),
        report_to="none",
        seed=seed,
        logging_steps=200,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds_full,
        eval_dataset=val_ds_full,
        data_collator=collator,
        tokenizer=tok,
        compute_metrics=compute_metrics,
        callbacks=[epoch_saver],
    )

    trainer.train()

    best_ckpt = trainer.state.best_model_checkpoint
    print("Best checkpoint:", best_ckpt)

    # Save clean BEST folder
    best_dir = OUTPUT_DIR / f"roberta_iemocap_final_seed{seed}_BEST"
    if best_dir.exists():
        shutil.rmtree(best_dir)
    shutil.copytree(best_ckpt, best_dir)
    tok.save_pretrained(best_dir)
    print("Saved BEST folder:", best_dir)

    test_metrics = trainer.evaluate(test_ds_full)
    print("TEST:", test_metrics)

    rows.append({
        "seed": seed,
        "best_dir": str(best_dir),
        "test_acc": float(test_metrics["eval_acc"]),
        "test_weighted_f1": float(test_metrics["eval_weighted_f1"]),
        "test_macro_f1": float(test_metrics["eval_macro_f1"]),
    })

df = pd.DataFrame(rows)
display(df)

print("\nMEAN:")
display(df[["test_acc", "test_weighted_f1", "test_macro_f1"]].mean().to_frame("mean"))

print("\nSTD:")
display(df[["test_acc", "test_weighted_f1", "test_macro_f1"]].std().to_frame("std"))


### Notes
- For GitHub, consider moving **training** into a script (`train.py`) so it can be run headlessly and used in CI.
- Keep large model checkpoints out of git (use releases, Hugging Face Hub, or external storage).
